In [ ]:
pip install pgmpy

  Dataset  : sklearn Breast Cancer Wisconsin (569 samples, 30 features)

  Model    : Discrete Bayesian Network
  
  Library  : pgmpy (latest), scikit-learn, matplotlib

LECTURE OUTLINE

* PART 1  DATA & PREPROCESSING   – load real data, discretise continuous features

* PART 2  REPRESENTATION          – understand what a Bayesian Network encodes

* PART 3  STRUCTURE LEARNING      – recover the DAG from data (HillClimb + BIC)

* PART 4  PARAMETER LEARNING      – fit CPDs via Maximum Likelihood Estimation

* PART 5  INFERENCE               – Variable Elimination for marginal & MAP queries

* PART 6  EVALUATION              – classification accuracy on a held-out test set


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import textwrap

import numpy  as np
import pandas as pd
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import networkx as nx

# scikit-learn
from sklearn.datasets         import load_breast_cancer
from sklearn.preprocessing    import KBinsDiscretizer
from sklearn.model_selection  import train_test_split
from sklearn.metrics          import (accuracy_score, confusion_matrix,
                                       classification_report)

# pgmpy
from pgmpy.models              import DiscreteBayesianNetwork
from pgmpy.estimators          import HillClimbSearch
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference           import VariableElimination


## PART 1 – DATA & PREPROCESSING

 **1-A  Load the Wisconsin Breast Cancer dataset from sklearn**

* 569 patients, 30 continuous cell-nucleus measurements

* Target: 0 = Malignant, 1 = Benign

In [ ]:
raw    = load_breast_cancer()
df_raw = pd.DataFrame(raw.data, columns=raw.feature_names)
df_raw["diagnosis"] = raw.target          # 0 = Malignant, 1 = Benign

print(f"Dataset shape  : {df_raw.shape}")
print(f"Class balance  : {dict(zip(['Malignant','Benign'], np.bincount(raw.target)))}")
print(f"\nFirst 3 rows (continuous):\n{df_raw.head(3).to_string()}\n")



Dataset shape  : (569, 31)
Class balance  : {'Malignant': np.int64(212), 'Benign': np.int64(357)}

First 3 rows (continuous):
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  mean compactness  mean concavity  mean concave points  mean symmetry  mean fractal dimension  radius error  texture error  perimeter error  area error  smoothness error  compactness error  concavity error  concave points error  symmetry error  fractal dimension error  worst radius  worst texture  worst perimeter  worst area  worst smoothness  worst compactness  worst concavity  worst concave points  worst symmetry  worst fractal dimension  diagnosis
0        17.99         10.38           122.8     1001.0          0.11840           0.27760          0.3001              0.14710         0.2419                 0.07871        1.0950         0.9053            8.589      153.40          0.006399            0.04904          0.05373               0.01587         0.03003                 0.006193     

**1-B  Feature selection**
  *  We keep 6 features that have high discriminatory power for a
 * readable graph. Students can extend this to all 30.

In [ ]:

FEATURES = [
    "mean radius",
    "mean texture",
    "mean concavity",
    "mean area",
    "worst radius",
    "worst concave points",
    ]
TARGET = "diagnosis"




**1-C  Discretisation**
* Bayesian Networks with discrete CPDs require categorical states.
* We bin each continuous feature into 3 quantile-equal buckets:
* "0.0" = Low   "1.0" = Medium   "2.0" = High

In [ ]:

kbd  = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
disc = kbd.fit_transform(df_raw[FEATURES])

df = pd.DataFrame(disc.astype(float).astype(str), columns=FEATURES)
df[TARGET] = df_raw[TARGET].astype(str).values   # "0" = Malignant, "1" = Benign

print("Discretised dataset (first 3 rows):")
print(df.head(3).to_string())
print(f"\nState space per variable:")
for col in df.columns:
    print(f"  {col:35s}: {sorted(df[col].unique())}")

# 1-D  Train / Test split (80 / 20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                     stratify=df[TARGET])
print(f"\nTrain size : {len(train_df)}   Test size : {len(test_df)}")

Discretised dataset (first 3 rows):
  mean radius mean texture mean concavity mean area worst radius worst concave points diagnosis
0         2.0          0.0            2.0       2.0          2.0                  2.0         0
1         2.0          1.0            1.0       2.0          2.0                  2.0         0
2         2.0          2.0            2.0       2.0          2.0                  2.0         0

State space per variable:
  mean radius                        : ['0.0', '1.0', '2.0']
  mean texture                       : ['0.0', '1.0', '2.0']
  mean concavity                     : ['0.0', '1.0', '2.0']
  mean area                          : ['0.0', '1.0', '2.0']
  worst radius                       : ['0.0', '1.0', '2.0']
  worst concave points               : ['0.0', '1.0', '2.0']
  diagnosis                          : ['0', '1']

Train size : 455   Test size : 114


## PART 2:  REPRESENTATION  (Bayesian Network)

In [ ]:
print(textwrap.dedent("""
A Bayesian Network (BN) is a Directed Acyclic Graph (DAG) where:
  • Nodes  = random variables  (features + diagnosis)
  • Edges  = direct probabilistic dependencies  A → B means A "influences" B
  • CPDs   = Conditional Probability Distributions P(node | parents)

The joint distribution factorises as:
  P(X₁, X₂, …, Xₙ) = ∏ P(Xᵢ | Pa(Xᵢ))

This compact representation encodes conditional independences via the
Markov condition and d-separation, enabling efficient inference even in
very large models.

Key components we will build step-by-step:
  1. Structure  (the DAG)
  2. Parameters (the CPD tables)
  3. Inference  (answering probabilistic queries)
"""))



A Bayesian Network (BN) is a Directed Acyclic Graph (DAG) where:
  • Nodes  = random variables  (features + diagnosis)
  • Edges  = direct probabilistic dependencies  A → B means A "influences" B
  • CPDs   = Conditional Probability Distributions P(node | parents)

The joint distribution factorises as:
  P(X₁, X₂, …, Xₙ) = ∏ P(Xᵢ | Pa(Xᵢ))

This compact representation encodes conditional independences via the
Markov condition and d-separation, enabling efficient inference even in
very large models.

Key components we will build step-by-step:
  1. Structure  (the DAG)
  2. Parameters (the CPD tables)
  3. Inference  (answering probabilistic queries)



## PART 3: STRUCTURE LEARNING – Hill-Climb + BIC

In [ ]:
print(textwrap.dedent("""
Structure learning discovers which edges to add/remove/reverse to best
explain the data.

Algorithm : Greedy Hill-Climbing Search
  • Start from an empty (or random) DAG
  • At each step try: add / remove / reverse every possible edge
  • Accept the change that maximises the scoring function
  • Stop when no improvement can be found

Scoring function : BIC (Bayesian Information Criterion)
  BIC(G, D) = log-likelihood(G, D) − (k/2) · log(n)

  The penalty term (k/2)·log(n) discourages overly complex graphs,
  acting as Occam's razor.  Higher BIC = better model.
"""))

hc             = HillClimbSearch(train_df)
learned_struct = hc.estimate(scoring_method="bic-d", max_indegree=3)
learned_edges  = list(learned_struct.edges())

print("Learned edges (DAG):")
for edge in learned_edges:
    print(f"  {edge[0]}  →  {edge[1]}")


Structure learning discovers which edges to add/remove/reverse to best
explain the data.

Algorithm : Greedy Hill-Climbing Search
  • Start from an empty (or random) DAG
  • At each step try: add / remove / reverse every possible edge
  • Accept the change that maximises the scoring function
  • Stop when no improvement can be found

Scoring function : BIC (Bayesian Information Criterion)
  BIC(G, D) = log-likelihood(G, D) − (k/2) · log(n)

  The penalty term (k/2)·log(n) discourages overly complex graphs,
  acting as Occam's razor.  Higher BIC = better model.



  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned edges (DAG):
  mean radius  →  mean area
  worst radius  →  mean radius
  worst concave points  →  mean concavity
  worst concave points  →  diagnosis
  diagnosis  →  worst radius
  diagnosis  →  mean texture


## PART 4 – PARAMETER LEARNING  (Maximum Likelihood Estimation)

In [ ]:
print(textwrap.dedent("""
Given the DAG structure, we estimate the CPD of each node from data.

Maximum Likelihood Estimation (MLE):
  P(Xᵢ = xᵢ | Pa(Xᵢ) = pa) = count(Xᵢ=xᵢ, Pa(Xᵢ)=pa) / count(Pa(Xᵢ)=pa)

This is simply a frequency count – the probability that a node takes a
particular value given its parents' values, estimated from the training data.
"""))

model = DiscreteBayesianNetwork(learned_edges)
model.fit(train_df, estimator=DiscreteMLE())

print(f"Number of nodes : {len(model.nodes())}")
print(f"Number of CPDs  : {len(model.cpds)}")
print("\nAll learned CPDs:\n" + "─" * 50)
for cpd in model.cpds:
    print(cpd)
    print()



Given the DAG structure, we estimate the CPD of each node from data.

Maximum Likelihood Estimation (MLE):
  P(Xᵢ = xᵢ | Pa(Xᵢ) = pa) = count(Xᵢ=xᵢ, Pa(Xᵢ)=pa) / count(Pa(Xᵢ)=pa)

This is simply a frequency count – the probability that a node takes a
particular value given its parents' values, estimated from the training data.

Number of nodes : 7
Number of CPDs  : 7

All learned CPDs:
──────────────────────────────────────────────────
+------------------+-----+----------------------+
| worst radius     | ... | worst radius(2.0)    |
+------------------+-----+----------------------+
| mean radius(0.0) | ... | 0.006622516556291391 |
+------------------+-----+----------------------+
| mean radius(1.0) | ... | 0.08609271523178808  |
+------------------+-----+----------------------+
| mean radius(2.0) | ... | 0.9072847682119205   |
+------------------+-----+----------------------+

+----------------+------------------+-----+----------------------+
| mean radius    | mean radius(0.0) | ...

## PART 5 – INFERENCE  (Variable Elimination)

In [ ]:
print(textwrap.dedent("""
Inference = answering probabilistic queries using the fitted model.

Variable Elimination (VE):
  • Marginal query  : P(Y | evidence)  – posterior distribution over Y
  • MAP query       : argmax_Y P(Y | evidence)  – most likely state of Y

VE works by summing out (marginalising) hidden variables one by one
using the factor representation of the CPDs, exploiting conditional
independence to avoid redundant computation.
"""))

infer = VariableElimination(model)




Inference = answering probabilistic queries using the fitted model.

Variable Elimination (VE):
  • Marginal query  : P(Y | evidence)  – posterior distribution over Y
  • MAP query       : argmax_Y P(Y | evidence)  – most likely state of Y

VE works by summing out (marginalising) hidden variables one by one
using the factor representation of the CPDs, exploiting conditional
independence to avoid redundant computation.



**5-A  Marginal query: prior probability of diagnosis (no evidence)**

In [ ]:

if TARGET in model.nodes():
    q_prior = infer.query([TARGET])
    print("Prior P(diagnosis) [no evidence]:")
    print(q_prior, "\n")




Prior P(diagnosis) [no evidence]:
+--------------+------------------+
| diagnosis    |   phi(diagnosis) |
+==============+==================+
| diagnosis(0) |           0.3736 |
+--------------+------------------+
| diagnosis(1) |           0.6264 |
+--------------+------------------+ 



**5-B  Posterior given clinical observations**

In [ ]:

evidence_1 = {}
if "mean radius"    in model.nodes(): evidence_1["mean radius"]    = "2.0"  # HIGH
if "mean concavity" in model.nodes(): evidence_1["mean concavity"] = "2.0"  # HIGH
if "mean area"      in model.nodes(): evidence_1["mean area"]      = "2.0"  # HIGH

if TARGET in model.nodes() and evidence_1:
    # remove TARGET from evidence if it snuck in
    evidence_1.pop(TARGET, None)
    q1 = infer.query([TARGET], evidence=evidence_1)
    print("Posterior P(diagnosis | high radius, high concavity, high area):")
    print(q1)
    print("  → 0 = Malignant, 1 = Benign\n")

evidence_2 = {}
if "mean radius"    in model.nodes(): evidence_2["mean radius"]    = "0.0"  # LOW
if "mean concavity" in model.nodes(): evidence_2["mean concavity"] = "0.0"  # LOW

if TARGET in model.nodes() and evidence_2:
    evidence_2.pop(TARGET, None)
    q2 = infer.query([TARGET], evidence=evidence_2)
    print("Posterior P(diagnosis | low radius, low concavity):")
    print(q2)
    print("  → 0 = Malignant, 1 = Benign\n")


Posterior P(diagnosis | high radius, high concavity, high area):
+--------------+------------------+
| diagnosis    |   phi(diagnosis) |
+==============+==================+
| diagnosis(0) |           0.9847 |
+--------------+------------------+
| diagnosis(1) |           0.0153 |
+--------------+------------------+
  → 0 = Malignant, 1 = Benign

Posterior P(diagnosis | low radius, low concavity):
+--------------+------------------+
| diagnosis    |   phi(diagnosis) |
+==============+==================+
| diagnosis(0) |           0.0027 |
+--------------+------------------+
| diagnosis(1) |           0.9973 |
+--------------+------------------+
  → 0 = Malignant, 1 = Benign



**5-C  MAP query**

In [ ]:

if TARGET in model.nodes() and evidence_1:
    map_result = infer.map_query([TARGET], evidence=evidence_1)
    label = "Malignant" if map_result[TARGET] == "0" else "Benign"
    print(f"MAP prediction for high-risk evidence: {label} (state={map_result[TARGET]})")

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

MAP prediction for high-risk evidence: Malignant (state=0)


## PART 6 – EVALUATION

In [ ]:
def predict(row: pd.Series) -> str:
    """Return MAP prediction for a single test row."""
    evidence = {
        feat: row[feat]
        for feat in FEATURES
        if feat in model.nodes() and feat != TARGET
    }
    try:
        result = infer.map_query([TARGET], evidence=evidence)
        return result[TARGET]
    except Exception:
        return "1"   # fallback: predict Benign if inference fails


y_true = test_df[TARGET].values
y_pred = [predict(row) for _, row in test_df.iterrows()]

acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred, labels=["0", "1"])

print(f"Accuracy : {acc:.4f}  ({acc*100:.1f}%)\n")
print("Confusion Matrix (rows = True, cols = Predicted):")
print(f"           Pred-Malignant  Pred-Benign")
print(f"  True-Malignant  {cm[0,0]:5d}          {cm[0,1]:5d}")
print(f"  True-Benign     {cm[1,0]:5d}          {cm[1,1]:5d}\n")
print("Full Classification Report:")
print(classification_report(y_true, y_pred,
                             target_names=["Malignant(0)", "Benign(1)"]))

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Accuracy : 0.9211  (92.1%)

Confusion Matrix (rows = True, cols = Predicted):
           Pred-Malignant  Pred-Benign
  True-Malignant     39              3
  True-Benign         6             66

Full Classification Report:
              precision    recall  f1-score   support

Malignant(0)       0.87      0.93      0.90        42
   Benign(1)       0.96      0.92      0.94        72

    accuracy                           0.92       114
   macro avg       0.91      0.92      0.92       114
weighted avg       0.92      0.92      0.92       114

